# LLM Baseline: Gemini 2.5 Flash Zero-Shot Classification

Phase 5 deliverable. Classifies the test split using Gemini 2.5 Flash with a plain-text
one-word instruction prompt. Results validated against a Pydantic `Literal` schema and
cached to MinIO so re-evaluation is cheap.

**Why plain-text instead of response_schema?**
`gemini-2.5-flash` is a thinking model. With `response_schema` + `response_mime_type`,
the SDK auto-parses `result.text` *before* the thinking tokens finish, returning `None`.
A direct "reply with ONE word" prompt is more reliable and still validates via Pydantic.

**Prerequisites**
- MinIO running (`docker compose up minio`)
- Vault running with real Gemini API key at `secret/gemini.api_key`
- `.env` at repo root

**Resume a partial run**: set `RESUME_RUN_ID` in Cell 1.

In [2]:
# Cell 1 — imports, env, clients
from __future__ import annotations

import io
import json
import os
import time
import uuid
from pathlib import Path
from typing import Literal

import httpx
import numpy as np
from dotenv import load_dotenv
from minio import Minio
from pydantic import BaseModel
from sklearn.metrics import accuracy_score, classification_report, f1_score

import google.genai as genai
from google.genai import types

REPO_ROOT = Path(".").resolve().parent
load_dotenv(REPO_ROOT / ".env", override=False)

# Set to a previous run_id to resume; None = start fresh
RESUME_RUN_ID: str | None = None

MINIO_ENDPOINT = os.environ["MINIO_LOCAL_ENDPOINT"].removeprefix("http://").removeprefix("https://")
secure = not (MINIO_ENDPOINT.startswith("localhost") or MINIO_ENDPOINT.startswith("127."))
minio  = Minio(MINIO_ENDPOINT, access_key=os.environ["MINIO_ACCESS_KEY"],
               secret_key=os.environ["MINIO_SECRET_KEY"], secure=secure)
BUCKET = os.environ["MINIO_BUCKET"]

# Vault fallback: try configured VAULT_ADDR, then localhost
vault_token = os.getenv("VAULT_ROOT_TOKEN", "dev-root-token")
for vault_addr in [os.getenv("VAULT_ADDR", "http://localhost:8200"), "http://localhost:8200"]:
    try:
        r = httpx.get(f"{vault_addr}/v1/secret/data/gemini",
                      headers={"X-Vault-Token": vault_token}, timeout=3.0)
        r.raise_for_status()
        GEMINI_KEY = r.json()["data"]["data"]["api_key"]
        break
    except Exception:
        continue
else:
    raise RuntimeError("Could not reach Vault")

assert not GEMINI_KEY.startswith("AIza-placeholder"), "Set a real Gemini key in Vault first"
gemini_client = genai.Client(api_key=GEMINI_KEY)
print("Clients ready")

Clients ready


In [3]:
# Cell 2 — schema and classify function
MODEL_ID = "gemini-2.5-flash"
LABELS   = ["bug", "feature", "docs", "question"]
VALID_LABELS: frozenset[str] = frozenset(LABELS)

# Cost (USD per 1M tokens, Gemini 2.5 Flash May 2026)
COST_INPUT_PER_M  = 0.075
COST_OUTPUT_PER_M = 0.30

_CLASSIFY_PROMPT = """\
Classify this GitHub issue as exactly one of: bug, feature, docs, question.

Definitions:
- bug: reports incorrect or unexpected behaviour, a crash, or a regression
- feature: requests new functionality, performance improvement, or refactoring
- docs: concerns only documentation, examples, or typos
- question: asks for usage help or clarification — no code change implied

Reply with ONLY the single word label, nothing else.

Issue:
{text}"""


class _LabelOutput(BaseModel):
    """Pydantic schema — validates that the parsed label is in the Literal union."""
    label: Literal["bug", "feature", "docs", "question"]


def classify_one(text: str) -> tuple[str, int, int]:
    """Return (predicted_label, input_tokens, output_tokens)."""
    response = gemini_client.models.generate_content(
        model=MODEL_ID,
        contents=_CLASSIFY_PROMPT.format(text=text[:3000]),
        config=types.GenerateContentConfig(temperature=0.0),
    )
    raw = (response.text or "").strip().lower()
    if raw in VALID_LABELS:
        label = raw
    else:
        found = next((lbl for lbl in VALID_LABELS if lbl in raw), None)
        if found is None:
            raise ValueError(f"Unexpected response: {repr(raw[:80])}")
        label = found
    validated = _LabelOutput(label=label)  # type: ignore[arg-type]
    usage = response.usage_metadata
    return validated.label, (usage.prompt_token_count or 0), (usage.candidates_token_count or 0)

print("Schema and classify function ready")

Schema and classify function ready


In [4]:
# Cell 3 — load test split and cache
def _download_split(split_name: str) -> list[dict]:
    resp = minio.get_object(BUCKET, f"splits/v1/{split_name}.jsonl")
    return [json.loads(line) for line in resp.read().decode("utf-8").splitlines() if line.strip()]

test_rows = _download_split("test")
print(f"Test split: {len(test_rows)} examples")

RUN_ID    = RESUME_RUN_ID or str(uuid.uuid4())
CACHE_KEY = f"models/llm_baseline/{RUN_ID}/predictions.json"
print(f"Run ID: {RUN_ID}")

try:
    cache_resp = minio.get_object(BUCKET, CACHE_KEY)
    cache: dict[str, str] = json.loads(cache_resp.read().decode("utf-8"))
    print(f"Loaded {len(cache)} cached predictions")
except Exception:
    cache = {}
    print("No cache — starting fresh")

Test split: 311 examples
Run ID: ee0e6070-6120-4c4e-9e0f-638153899e47
No cache — starting fresh


In [5]:
# Cell 4 — inference (skips cached rows)
y_true: list[str]   = []
y_pred: list[str]   = []
latencies: list[float] = []
total_in  = total_out = errors = 0

for i, row in enumerate(test_rows):
    issue_id = str(row["id"])
    y_true.append(row["label"])

    if issue_id in cache:
        y_pred.append(cache[issue_id])
        continue

    t0 = time.perf_counter()
    try:
        label, in_tok, out_tok = classify_one(row["text"])
    except Exception as exc:
        print(f"  WARNING issue {issue_id}: {exc}")
        label, in_tok, out_tok = "bug", 0, 0
        errors += 1

    latency_ms = (time.perf_counter() - t0) * 1000
    latencies.append(latency_ms)
    total_in  += in_tok
    total_out += out_tok
    y_pred.append(label)
    cache[issue_id] = label

    if (i + 1) % 25 == 0:
        payload = json.dumps(cache, indent=2).encode("utf-8")
        minio.put_object(BUCKET, CACHE_KEY, io.BytesIO(payload),
                         length=len(payload), content_type="application/json")
        print(f"[{i+1}/{len(test_rows)}]  last={latency_ms:.0f}ms")

payload = json.dumps(cache, indent=2).encode("utf-8")
minio.put_object(BUCKET, CACHE_KEY, io.BytesIO(payload), length=len(payload), content_type="application/json")
print(f"Done. Errors: {errors}")

[25/311]  last=2576ms
[50/311]  last=2083ms
[75/311]  last=3202ms
[100/311]  last=2643ms
[125/311]  last=2468ms
[150/311]  last=3078ms
[175/311]  last=3237ms
[200/311]  last=3902ms
[225/311]  last=4754ms
[250/311]  last=2624ms
[275/311]  last=2123ms
[300/311]  last=4020ms
Done. Errors: 0


In [6]:
# Cell 5 — metrics and cost
accuracy  = accuracy_score(y_true, y_pred)
macro_f1  = f1_score(y_true, y_pred, average="macro", labels=LABELS, zero_division=0)
per_class = f1_score(y_true, y_pred, average=None, labels=LABELS, zero_division=0)
per_class_dict = dict(zip(LABELS, per_class.tolist()))

p50 = float(np.percentile(latencies, 50)) if latencies else 0.0
p95 = float(np.percentile(latencies, 95)) if latencies else 0.0

cost_total  = total_in  / 1_000_000 * COST_INPUT_PER_M + total_out / 1_000_000 * COST_OUTPUT_PER_M
cost_per_1k = cost_total / len(test_rows) * 1000

print(f"Test accuracy  : {accuracy:.4f}")
print(f"Test macro-F1  : {macro_f1:.4f}")
for cls, score in per_class_dict.items():
    print(f"  {cls:<10} F1 = {score:.4f}")
print()
print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))
print(f"Latency p50 : {p50:.1f} ms  p95 : {p95:.1f} ms  (n={len(latencies)} live calls)")
print(f"Tokens      : {total_in:,} in  {total_out:,} out")
print(f"Cost total  : ${cost_total:.5f}  Cost/1k : ${cost_per_1k:.5f}")

Test accuracy  : 0.9325
Test macro-F1  : 0.6853
  bug        F1 = 0.9585
  feature    F1 = 0.8776
  docs       F1 = 0.9051
  question   F1 = 0.0000

              precision    recall  f1-score   support

         bug       0.92      1.00      0.96       185
     feature       0.91      0.84      0.88        51
        docs       0.98      0.84      0.91        74
    question       0.00      0.00      0.00         1

    accuracy                           0.93       311
   macro avg       0.70      0.67      0.69       311
weighted avg       0.93      0.93      0.93       311

Latency p50 : 2737.3 ms  p95 : 5620.7 ms  (n=311 live calls)
Tokens      : 245,154 in  311 out
Cost total  : $0.01848  Cost/1k : $0.05942
